# 03 — Timing Attacks

## What This Is
Timing attacks exploit secret-dependent execution time differences. Paul Kocher's 1996 timing attack on RSA showed that measuring decryption time leaks the private key. Remote timing attacks work over network connections.

## Why It Matters
Cache-timing attacks (FLUSH+RELOAD, PRIME+PROBE) break AES on OpenSSL, extract RSA keys, and defeat ASLR. Meltdown/Spectre use timing to exfiltrate speculatively-accessed memory. Constant-time programming is now mandatory for any secret-dependent computation.

## Where the Community Stands
Constant-time crypto is the industry standard (libsodium, BoringSSL, rustcrypto). compilers do not guarantee constant-time even from constant-time source — hardware (branch predictors, memory subsystems) adds timing variance.

## Key Vulnerability Pattern
Any code with an early-exit on secret-dependent comparison is vulnerable:
```python
# INSECURE: timing reveals which byte mismatches
if secret[i] != provided[i]:
    return False  # early exit!
# SECURE: constant-time compare (all bytes always compared)
```

In [ ]:
import time
import random
import statistics
import hmac
import hashlib
from typing import List, Tuple

# ---- Vulnerable string comparison ----
def insecure_compare(a: bytes, b: bytes) -> bool:
    """VULNERABLE: exits on first mismatch — leaks how many bytes match."""
    if len(a) != len(b):
        return False
    for i in range(len(a)):
        if a[i] != b[i]:
            return False
    return True

def constant_time_compare(a: bytes, b: bytes) -> bool:
    """SECURE: always compares all bytes (same as hmac.compare_digest)."""
    if len(a) != len(b):
        return False
    result = 0
    for x, y in zip(a, b):
        result |= x ^ y
    return result == 0

SECRET_TOKEN = b'super-secret-api-key-12345678901'
print(f'Secret token: {SECRET_TOKEN}')
print(f'Length: {len(SECRET_TOKEN)} bytes')

In [ ]:
def measure_time_ns(fn, *args) -> float:
    start = time.perf_counter_ns()
    fn(*args)
    return time.perf_counter_ns() - start

def timing_attack_simulation(secret: bytes, compare_fn,
                              n_samples: int = 100) -> bytes:
    """Byte-by-byte timing attack against a comparison function."""
    recovered = bytearray(len(secret))
    alphabet  = list(range(256))

    for pos in range(len(secret)):
        times = {}
        for byte_val in alphabet:
            candidate = bytearray(recovered)
            candidate[pos] = byte_val
            # Pad rest with 0 (will mismatch early for wrong prefix)
            t_list = []
            for _ in range(n_samples):
                t = measure_time_ns(compare_fn, secret, bytes(candidate))
                t_list.append(t)
            times[byte_val] = statistics.median(t_list)
        best = max(times, key=lambda b: times[b])
        recovered[pos] = best

    return bytes(recovered)

# Demonstrate on a short token for speed
short_secret = b'KEY4'
print('Timing attack against insecure_compare:')
rng = random.Random(42)
recovered = timing_attack_simulation(short_secret, insecure_compare, n_samples=50)
print(f'  Target:    {short_secret}')
print(f'  Recovered: {recovered}')
print(f'  Success:   {recovered == short_secret}')

In [ ]:
# Verify constant-time compare resists timing
print('\nTiming profiling: median ns per comparison')
test_cases = [
    (b'\x00' * 4, 'All wrong (0 bytes match)'),
    (b'KEY\x00',  '3 bytes match'),
    (b'KEY4',     'Correct (all match)'),
]

for impl_name, fn in [('insecure_compare', insecure_compare), ('constant_time', constant_time_compare)]:
    print(f'\n  {impl_name}:')
    for guess, label in test_cases:
        times = [measure_time_ns(fn, short_secret, guess) for _ in range(500)]
        print(f'    {label:<30} median={statistics.median(times):.0f}ns  stdev={statistics.stdev(times):.0f}ns')

# Python's built-in solution
print('\n  hmac.compare_digest (Python stdlib):')
for guess, label in test_cases:
    times = [measure_time_ns(hmac.compare_digest, short_secret, guess) for _ in range(500)]
    print(f'    {label:<30} median={statistics.median(times):.0f}ns')

## Cache Timing: FLUSH+RELOAD Concept
Flush+Reload is a cache side-channel that detects whether a specific memory address was accessed by another process. It underpins Spectre/Meltdown exploits. The attacker flushes a cache line, waits for the victim to potentially access it, then reloads — fast reload = victim accessed it.

In [ ]:
# Conceptual simulation of cache timing oracle
# Real implementation requires clflush instruction (not available in Python sandbox)

class SimulatedCacheOracle:
    """Simulates a last-level cache with hit/miss timing."""

    CACHE_HIT_NS  = 5      # ~5ns for L1 hit
    CACHE_MISS_NS = 200    # ~200ns for DRAM access
    CACHE_SIZE    = 64     # number of 64-byte lines we track

    def __init__(self, rng: random.Random):
        self.rng        = rng
        self.cache: set = set()

    def flush(self, addr: int) -> None:
        self.cache.discard(addr)

    def load(self, addr: int) -> int:
        """Returns simulated access time (ns)."""
        if addr in self.cache:
            noise = self.rng.gauss(0, 1)
            return max(1, int(self.CACHE_HIT_NS + noise))
        else:
            # Evict random line to maintain capacity
            if len(self.cache) >= self.CACHE_SIZE:
                self.cache.discard(next(iter(self.cache)))
            self.cache.add(addr)
            noise = self.rng.gauss(0, 10)
            return max(100, int(self.CACHE_MISS_NS + noise))

# Simulate FLUSH+RELOAD attack
rng = random.Random(0)
oracle = SimulatedCacheOracle(rng)

SECRET_INDEX = 42  # secret table index (simulates speculative load)
PROBE_ADDRS  = list(range(0, 256, 16))  # one per 4KB page stride

def victim_access(secret_idx: int, cache: SimulatedCacheOracle) -> None:
    """Victim loads from a table indexed by secret."""
    addr = PROBE_ADDRS[secret_idx % len(PROBE_ADDRS)]
    cache.load(addr)

print('FLUSH+RELOAD simulation:')
flush_reload_results = {}

for probe in PROBE_ADDRS:
    oracle.flush(probe)

victim_access(SECRET_INDEX, oracle)  # victim accesses secret-indexed addr

for i, probe in enumerate(PROBE_ADDRS):
    t = oracle.load(probe)
    flush_reload_results[i*16] = t

# Find minimum time (cache hit = accessed by victim)
min_probe = min(flush_reload_results, key=flush_reload_results.get)
min_time  = flush_reload_results[min_probe]

print(f'  Cache hit threshold: <{SimulatedCacheOracle.CACHE_HIT_NS*3}ns')
for idx, t in sorted(flush_reload_results.items())[:8]:
    hit = '  <-- HIT (secret accessed here!)' if t <= SimulatedCacheOracle.CACHE_HIT_NS*3 else ''
    print(f'  Probe[{idx:3d}]: {t:4d}ns{hit}')
print(f'\n  Inferred secret index: {min_probe//16}  True: {SECRET_INDEX}')